In [2]:
!pip install ecdsa


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip


In [3]:
from ecdsa import SigningKey, SECP256k1,VerifyingKey
import hashlib

def sha256_hex(b):
    return hashlib.sha256(b).hexdigest()
def generate_wallet():
    sk = SigningKey.generate(curve=SECP256k1)
    vk = sk.get_verifying_key()
    assert vk is not None
    address = sha256_hex(vk.to_string())[:40]
    return sk, vk, address

In [4]:
class TxOutput:
    def __init__(self, value, recipient):
        self.value = value
        self.recipient = recipient
    def to_dict(self):
        return {"value": self.value, "recipient": self.recipient}
class TxInput:
    def __init__(self, prev_tx, prev_index, signature="", pubkey=""):
        self.prev_tx = prev_tx
        self.prev_index = prev_index
        self.signature = signature
        self.pubkey = pubkey

    def to_dict(self):
        return {
            "prev_tx": self.prev_tx,
            "prev_index": self.prev_index,
            "pubkey": self.pubkey,
        }
class Transaction:
    def __init__(self, inputs, outputs):
        self.inputs = inputs
        self.outputs = outputs
    def to_dict(self):
        return {
            "inputs": [inp.to_dict() for inp in self.inputs],
            "outputs": [out.to_dict() for out in self.outputs]
        }
class UTXOPool:
    def __init__(self):
        self.utxos = {} # (tx_id, index): TxOutput

In [5]:
import json, hashlib
def tx_hash(tx):
    return hashlib.sha256(json.dumps(tx,sort_keys=True).encode()).hexdigest()
def sign(sk, message):
    return sk.sign(message.encode()).hex()
def verify(vk, signature_hex, message):
    try:
        vk.verify(bytes.fromhex(signature_hex), message.encode())
        return True
    except:
        return False

In [6]:
def validate_and_apply_transaction(tx: Transaction, utxos: dict) -> bool:
    total_in = 0
    tx_digest = tx_hash(tx.to_dict())

    for inp in tx.inputs:
        key = (inp.prev_tx, inp.prev_index)
        utxo = utxos.get(key)
        if not utxo:
            print("Invalid input: spent or unknown UTXO")
            return False

        addr = sha256_hex(bytes.fromhex(inp.pubkey))[:40]
        if addr != utxo.recipient:
            print("Signature does not match owner.")
            return False

        vk = VerifyingKey.from_string(bytes.fromhex(inp.pubkey), curve=SECP256k1)
        if not verify(vk, inp.signature, tx_digest):
            print("Invalid signature")
            return False

        total_in += utxo.value

    total_out = sum(o.value for o in tx.outputs)
    if total_out > total_in:
        print("Overspending!")
        return False

    # 2. Update UTXO Pool (Spend old, create new)
    for inp in tx.inputs:
        del utxos[(inp.prev_tx, inp.prev_index)]

    new_txid = tx_digest
    for idx, out in enumerate(tx.outputs):
        utxos[(new_txid, idx)] = out

    return True

In [7]:
def create_transaction(inputs_info, outputs_info, sk, vk):
    tx_inputs = [TxInput(prev_tx=h, prev_index=idx, pubkey=vk.to_string().hex()) for h, idx in inputs_info]
    tx_outputs = [TxOutput(val, recip) for val, recip in outputs_info]
    tx = Transaction(tx_inputs, tx_outputs)

    message = tx_hash(tx.to_dict())
    signature = sign(sk, message)
    for inp in tx.inputs:
        inp.signature = signature

    return tx

In [8]:
tarek_sk, tarek_vk, tarek_addr = generate_wallet()
dahmen_sk, dahmen_vk, dahmen_addr = generate_wallet()
carol_sk, carol_vk, carol_addr = generate_wallet()

# Coinbase: gives Tarek 50 coins
coinbase_tx = {
    "inputs": [],
    "outputs": [{"value": 50, "recipient": tarek_addr}]
}
coinbase_id = tx_hash(coinbase_tx)
utxos = {(coinbase_id, 0): TxOutput(50, tarek_addr)}

# Tarek sends 30 to Dahmen and keeps 20 change
tx1 = create_transaction(
    inputs_info=[(coinbase_id, 0)],
    outputs_info=[(30, dahmen_addr), (20, tarek_addr)],
    sk=tarek_sk,
    vk=tarek_vk
)
print("Transaction 1: tarek -> dahmen (30), Change: 20", "Pass" if validate_and_apply_transaction(tx1, utxos) else "Failed")

# Dahmen sends 10 to Carol, 20 change to himself
tx1_id = tx_hash(tx1.to_dict())
tx2 = create_transaction(
    inputs_info=[(tx1_id, 0)],  # Dahmen spends output 0 (value 30)
    outputs_info=[(10, carol_addr), (20, dahmen_addr)],
    sk=dahmen_sk,
    vk=dahmen_vk
)
print("Transaction 2: dahmen -> Carol (10), Change: 20", "Pass" if validate_and_apply_transaction(tx2, utxos) else "Failed")

# Tarek attempts to double-spend the original coinbase UTXO
tx3 = create_transaction(
    inputs_info=[(coinbase_id, 0)],  # Already spent in tx1
    outputs_info=[(30, dahmen_addr), (20, tarek_addr)],
    sk=tarek_sk,
    vk=tarek_vk
)
is_valid = validate_and_apply_transaction(tx3, utxos)
print("Transaction 3: tarek double-spend attempt", "Pass (Incorrectly Accepted)" if is_valid else "Blocked as expected")

Transaction 1: tarek -> dahmen (30), Change: 20 Pass
Transaction 2: dahmen -> Carol (10), Change: 20 Pass
Invalid input: spent or unknown UTXO
Transaction 3: tarek double-spend attempt Blocked as expected
